# 02 - Stress Period Selection

Synthetic engineering and validation evidence, not final regulatory capital.

Use this notebook with the [IMA package journey](../docs/PACKAGE_JOURNEY.md) and the executable [package demo](../examples/run_demo.py).

Raw inputs your upstream risk engine must emit: scenario P&L cubes with scenario metadata, risk-factor definitions, RFET real-price observations and evidence, stress-history series, PLA/backtesting observations, and NMRF valuation artifacts. The package dataset contract defines the committed fixture files, manifest lineage, and Arrow handoff: [`DATASET_CONTRACT.md`](../docs/DATASET_CONTRACT.md). The staged workflow is described in the [IMA package journey](../docs/PACKAGE_JOURNEY.md).

This notebook demonstrates model-validation review of the supplied historical risk-class loss histories in the committed `capital_run_v1` fixture. It uses the fixture through the shared loader in `examples.capital_run_fixture`; no separate synthetic data is created here.

Regulatory anchors: Basel MAR33 stress-period and NMRF stress-scenario concepts; U.S. NPR 2.0 proposed-rule parameters for stressed ES / SES calibration cited below; EU CRR Articles 325bc and 325bk as comparison context.

Prototype caution: outputs are deterministic development evidence only. They are not final regulatory capital or supervisory approval evidence.


## Flow

```mermaid
flowchart LR
    A[Historical risk-class loss series] --> B[Rolling policy windows]
    B --> C[Severity score calculation]
    C --> D[Selected stress period by risk class]
    D --> E[Distribution diagnostics]
    E --> F[Stress-period validation evidence]
```


## Public API happy path

This notebook selects stress periods from historical loss series using top-level IMA stress-period APIs.


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
for path in (ROOT, ROOT / "src"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display

from examples.capital_run_fixture import (
    as_of_date_from_fixture,
    load_capital_run_v1_fixture,
    policy_from_fixture,
)
from examples.notebook_utils import IMA_CHART_COLORS, apply_notebook_plot_style
from frtb_ima import (
    rolling_window_severity_scores,
    select_stress_periods_for_policy,
)

apply_notebook_plot_style()

fixture = load_capital_run_v1_fixture()
policy = policy_from_fixture(fixture)
as_of_date = as_of_date_from_fixture(fixture)

selection = select_stress_periods_for_policy(
    fixture.stress_histories,
    policy,
    as_of_date=as_of_date,
    run_id=str(fixture.params["run_id"]),
    desk_id=str(fixture.params["desk_id"]),
)

display(
    Markdown(
        f"Loaded `{fixture.root.name}` for `{policy.regime.value}` as of "
        f"`{as_of_date.isoformat()}`. Stress window: "
        f"`{selection.window_observations}` observations."
    )
)


## Implementation details / math verification - Rolling severity and selection diagnostics

The remaining cells inspect rolling severity curves and distribution checks for selected stress periods.


## Selection Summary

The prototype selects one common stress window per risk class from supplied positive-loss histories. The severity score is the empirical ES tail mean for each rolling policy window. The selected period is the highest-severity candidate under the deterministic tie-break rule.

In [ ]:
def markdown_table(headers: list[str], rows: list[list[str]]) -> Markdown:
    header = "| " + " | ".join(headers) + " |"
    separator = "| " + " | ".join(["---"] * len(headers)) + " |"
    body = ["| " + " | ".join(row) + " |" for row in rows]
    return Markdown("\n".join([header, separator, *body]))


summary_rows: list[list[str]] = []
histories_by_risk_class = {series.risk_class: series for series in fixture.stress_histories}
for risk_class in sorted(selection.selected_by_risk_class, key=lambda item: item.value):
    candidate = selection.selected_by_risk_class[risk_class]
    series = histories_by_risk_class[risk_class]
    summary_rows.append(
        [
            risk_class.value,
            f"{series.observation_count}",
            f"{selection.candidate_counts[risk_class]}",
            candidate.start_date.isoformat(),
            candidate.end_date.isoformat(),
            f"{candidate.severity_score:,.2f}",
            candidate.period_id,
        ]
    )

display(
    markdown_table(
        [
            "Risk class",
            "Observations",
            "Candidate windows",
            "Selected start",
            "Selected end",
            "Severity score",
            "Period ID",
        ],
        summary_rows,
    )
)


## Rolling Severity Curves

These plots show every rolling policy-window severity score. The shaded band marks the selected stress window for the same risk class. This lets validators see whether the chosen window is a clear peak or a marginal tie-sensitive result.

In [ ]:
histories = tuple(sorted(fixture.stress_histories, key=lambda series: series.risk_class.value))
fig, axes = plt.subplots(len(histories), 1, figsize=(11, 10), sharex=True)

for ax, series in zip(np.atleast_1d(axes), histories, strict=True):
    scores = rolling_window_severity_scores(
        series.losses,
        window_observations=selection.window_observations,
        minimum_observations=selection.minimum_observations,
        severity_metric=selection.severity_metric,
        confidence_level=selection.confidence_level,
        es_estimator=selection.es_estimator,
    )
    window_end_dates = series.dates[selection.window_observations - 1 :]
    candidate = selection.selected_by_risk_class[series.risk_class]

    ax.plot(window_end_dates, scores, color=IMA_CHART_COLORS["blue"], linewidth=1.4)
    ax.axvspan(candidate.start_date, candidate.end_date, color=IMA_CHART_COLORS["amber"], alpha=0.18)
    ax.scatter(
        [candidate.end_date],
        [candidate.severity_score],
        color=IMA_CHART_COLORS["red"],
        s=28,
        zorder=3,
        label="Selected window end",
    )
    ax.set_ylabel(series.risk_class.value)
    ax.margins(x=0.01)

axes[0].legend(loc="upper left", frameon=False)
fig.suptitle("Rolling stress-window severity by risk class", y=0.995)
fig.supxlabel("Window end date")
fig.supylabel("ES severity score, positive = loss")
fig.tight_layout()


## Severity Distribution Check

A selected stress window should be understood relative to the available candidate distribution. This chart compares the selected severity with the median and 90th percentile rolling-window severity for each risk class.

In [ ]:
risk_labels: list[str] = []
median_scores: list[float] = []
p90_scores: list[float] = []
selected_scores: list[float] = []

for series in histories:
    scores = rolling_window_severity_scores(
        series.losses,
        window_observations=selection.window_observations,
        minimum_observations=selection.minimum_observations,
        severity_metric=selection.severity_metric,
        confidence_level=selection.confidence_level,
        es_estimator=selection.es_estimator,
    )
    risk_labels.append(series.risk_class.value)
    median_scores.append(float(np.median(scores)))
    p90_scores.append(float(np.percentile(scores, 90)))
    selected_scores.append(selection.selected_by_risk_class[series.risk_class].severity_score)

x = np.arange(len(risk_labels))
width = 0.25
fig, ax = plt.subplots(figsize=(10, 4.8))
ax.bar(x - width, median_scores, width=width, label="Median", color=IMA_CHART_COLORS["dark_gray"])
ax.bar(x, p90_scores, width=width, label="90th percentile", color=IMA_CHART_COLORS["blue"])
ax.bar(x + width, selected_scores, width=width, label="Selected", color=IMA_CHART_COLORS["red"])
ax.set_xticks(x, risk_labels)
ax.set_ylabel("Severity score, positive = loss")
ax.set_title("Selected stress severity versus candidate distribution")
ax.legend(frameon=False)
fig.tight_layout()


## See also

- [IMA package journey](../docs/PACKAGE_JOURNEY.md)
- [IMA dataset contract](../docs/DATASET_CONTRACT.md)
- [Client integration guide](../../../docs/CLIENT_INTEGRATION.md)
- [SBM package journey](../../frtb-sbm/docs/PACKAGE_JOURNEY.md)
- [DRC package journey](../../frtb-drc/docs/PACKAGE_JOURNEY.md)
- [RRAO package journey](../../frtb-rrao/docs/PACKAGE_JOURNEY.md)
- [CVA package journey](../../frtb-cva/docs/PACKAGE_JOURNEY.md)
